# Stage 0 · Data and environment

Goal of this notebook: create every input the later stages need, reproducibly.

1. **Intent data** (Bitext customer support): load, inspect, clean, deduplicate, split into train / dev / test without leakage.
2. **Order database** (SQLite): customers, products (from WANDS), orders, refunds, plus hand-made edge cases on policy boundaries.
3. **A data summary** in `results/` so every number in the README can be traced back.

> یادداشت: هر عدد این نوت‌بوک با `SEED` و `reference_date` ثابت تولید می‌شود؛ اگر دوباره اجرا کنی باید دقیقاً همان خروجی‌ها را بگیری. توضیح کامل هر تصمیم در صفحه‌ی آموزشی مرحله‌ی ۰ است.

In [ ]:
import json, sqlite3, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from support import config
from support.data import clean_intents, load_bitext, load_wands_products, split_intents

config.ensure_dirs()
pd.set_option("display.max_colwidth", 90)
print("project root:", config.ROOT)
print("reference date (simulated today):", config.reference_date())

## Part A · Intent data

Bitext is downloaded from the Hugging Face Hub once and cached in `data/raw/bitext.csv`.
If the download fails, download the CSV by hand from the dataset page and pass its path to `load_bitext("path/to/file.csv")`.

In [ ]:
raw = load_bitext()
print(raw.shape)
raw.head()

### Look before you model
Three questions: how balanced are the intents, how long are the messages, and which language phenomena (the `flags` column) are present.
The flags matter later: **Z = typos**, **Q = colloquial**, **K = keyword mode** are exactly the "noisy inputs" the job ad talks about, so we will report accuracy per flag in stage 1 and stage 7.

In [ ]:
counts = raw["intent"].value_counts()
print(f"{raw['intent'].nunique()} intents, {raw['category'].nunique()} categories")
print("rows per intent: min", counts.min(), "| max", counts.max())

lengths = raw["instruction"].str.split().str.len()
print("\nwords per message:")
print(lengths.describe().round(1).to_string())

flag_share = (raw["flags"].fillna("").apply(set).explode().value_counts() / len(raw)).round(3)
print("\nshare of rows with each flag:")
print(flag_share.to_string())

In [ ]:
ax = counts.sort_values().plot.barh(figsize=(7, 8), title="Messages per intent")
ax.set_xlabel("rows");

### Does our routing table cover every intent?
`configs/intents.yaml` decides whether an intent goes to a tool, to policy RAG, to a human, or is out of scope.
If Bitext contains an intent the file does not know, we want to find out **now**, not when the agent crashes.

In [ ]:
known = set(config.intent_map())
present = set(raw["intent"].unique())
print("missing from intents.yaml:", sorted(present - known) or "none")
print("in intents.yaml but not in data:", sorted(known - present) or "none")

routes = pd.Series({i: config.intent_map()[i]["route"] for i in present & known})
print("\nintents per route:")
print(routes.value_counts().to_string())

### Clean and deduplicate
Bitext is generated from templates, so many messages differ only in a placeholder or punctuation.
`normalize()` builds a key that ignores those differences. We drop exact duplicates and any key that carries two different intents (an ambiguous label).

In [ ]:
clean, report = clean_intents(raw)
print(json.dumps(report, indent=2))

top = clean["key"].value_counts().head(5)
print("\nmost repeated keys (same sentence, different surface form):")
for key, n in top.items():
    print(f"{n:>3} x  {key}")
    print("      e.g.", clean.loc[clean['key'] == key, 'text'].head(2).tolist())

### Split without leakage
A random split would put "cancel order {{Order Number}}" in train and "Cancel order {{Order ID}}!" in test: the test score would measure memory, not understanding.
`split_intents()` uses `StratifiedGroupKFold`: grouped by key (no key in two splits) and stratified by intent (similar class balance everywhere).

In [ ]:
splits = split_intents(clean)
for name, part in splits.items():
    print(f"{name:>5}: {len(part):>6} rows  ({len(part) / len(clean):.0%})")

share = pd.DataFrame({n: p["intent"].value_counts(normalize=True) for n, p in splits.items()}).fillna(0)
print("\nlargest difference in an intent's share between splits:",
      f"{(share.max(axis=1) - share.min(axis=1)).max():.3%}")

for name, part in splits.items():
    part.to_csv(config.PROCESSED_DIR / f"intents_{name}.csv", index=False)
print("saved to", config.PROCESSED_DIR)

> **The test split is now locked.** Use `dev` for every decision (model choice, thresholds, prompts). Open `test` once per stage, at the end, to report the final number.

## Part B · Order database

WANDS products give us realistic product names. If you already downloaded WANDS for the product-search project,
point `WANDS_DIR` at that folder to reuse it; otherwise it is downloaded once.

In [ ]:
import os
from support.orders import build_database

WANDS_DIR = os.getenv("WANDS_DIR")          # e.g. "../../product-search-rag/data/wands"
wands = load_wands_products(WANDS_DIR)
print(wands.shape)

sizes = build_database(wands, config.DB_PATH)
print(sizes)

### Sanity checks with plain SQL
The agent will only ever see this data through tools, so it must be right. The same checks run automatically in `tests/test_orders.py`.

In [ ]:
con = sqlite3.connect(config.DB_PATH)
q = lambda sql: pd.read_sql(sql, con)

print(q("SELECT status, COUNT(*) AS n, ROUND(AVG(total_eur), 2) AS avg_eur FROM orders GROUP BY status"))
print()
print(q("SELECT status, COUNT(*) AS n FROM refunds GROUP BY status"))

In [ ]:
q("""
SELECT o.order_id, o.scenario, o.status, o.total_eur, o.shipped_at, o.delivered_at, p.custom_made
FROM orders o JOIN products p USING (product_id)
WHERE o.scenario IS NOT NULL
ORDER BY o.order_id
""")

In [ ]:
# What a support agent would see for one customer: this is the shape of a future tool result.
q("""
SELECT c.name, c.email, c.language, o.order_id, o.status, o.total_eur, p.name AS product
FROM customers c JOIN orders o USING (customer_id) JOIN products p USING (product_id)
WHERE c.customer_id = 'C0002'
""")

## Data summary
One JSON file with every number from this notebook. The README and later stages read from here, never from memory.

In [ ]:
import platform, sklearn

summary = {
    "reference_date": str(config.reference_date()),
    "seed": config.SEED,
    "bitext": {"rows_raw": len(raw), "intents": int(raw["intent"].nunique()), **report,
               "splits": {n: len(p) for n, p in splits.items()}},
    "database": sizes,
    "orders_by_status": q("SELECT status, COUNT(*) n FROM orders GROUP BY status").set_index("status")["n"].to_dict(),
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "sklearn": sklearn.__version__},
}
(config.RESULTS_DIR / "data_summary.json").write_text(json.dumps(summary, indent=2))
con.close()
print(json.dumps(summary, indent=2))

## Before moving on
Run the tests from the project root:

```bash
pytest -q
```

All green means: dates follow the order lifecycle, every status and edge case exists, refunds respect the auto-approval limit, the build is deterministic, the splits share no sentence, and the English and German policies state the same numbers as `policy_rules.yaml`.